# Notebook 02 — Preprocessing & Augmentation Pipeline
## Edge AI Acoustic Border Intrusion Detection

---

### Context from EDA (Notebook 01)
| Issue Identified | Action in This Notebook |
|-----------------|------------------------|
| 40 footstep files at wrong SR | Force resample all → 16 kHz |
| 15 clipped files (balastic+noise) | Peak normalisation |
| 1 silent noise file | Flag and remove |
| Footstep kurtosis = 345 (sparse transients) | Energy-VAD gating before feature extraction |
| Class imbalance 4.68:2.60:1.00 | Augmentation → target 320 footstep samples |
| C0/C1 dominate MFCC dynamics | Extract 8 MFCCs + Δ + ΔΔ = 24-dim vector |
| Variable durations (3–5s) | Hard trim/pad to exactly 3.0s |

### Pipeline Stages

In [ ]:
import os, shutil, warnings, random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from collections import defaultdict

import librosa
import librosa.display
import soundfile as sf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
import seaborn as sns

warnings.filterwarnings("ignore")
random.seed(42)
np.random.seed(42)

# ── Research Plot Style ──────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":       "serif",
    "font.serif":        ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size":         12,
    "axes.labelsize":    13,
    "axes.titlesize":    13,
    "axes.linewidth":    1.2,
    "xtick.labelsize":   11,
    "ytick.labelsize":   11,
    "xtick.major.size":  6,
    "ytick.major.size":  6,
    "xtick.minor.size":  3,
    "ytick.minor.size":  3,
    "legend.fontsize":   11,
    "legend.frameon":    True,
    "legend.edgecolor":  "0.4",
    "grid.linestyle":    ":",
    "grid.linewidth":    0.7,
    "grid.alpha":        0.85,
    "figure.dpi":        150,
    "savefig.dpi":       300,
    "savefig.bbox":      "tight",
})

def paper_axes(ax):
    ax.minorticks_on()
    ax.grid(True, which="major", linestyle=":", linewidth=0.8)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.6)
    for sp in ax.spines.values():
        sp.set_linewidth(1.2)
    ax.tick_params(which="both", direction="in", top=True, right=True)

CLASS_COLORS = {
    "footsteps": "#2166ac",
    "noise":     "#d6604d",
    "balastic":  "#4dac26",
}

# ── Pipeline Config ──────────────────────────────────────────────────────────
DATASET_PATH    = Path("/kaggle/input/datasets/katakuricharlotte/borderintrusiondetection-data/dataset")
OUTPUT_PATH     = Path("/kaggle/working/processed")
FIGURES_PATH    = Path("/kaggle/working/figures")
PROCESSED_AUDIO = OUTPUT_PATH / "audio_clean"

SR              = 16000
CLIP_DURATION   = 3.0
CLIP_SAMPLES    = int(SR * CLIP_DURATION)

# MFCC config (informed by EDA: C0-C7 sufficient)
N_MFCC          = 8
N_FFT           = 512
HOP_LENGTH      = 256
N_MELS          = 40
FEATURE_DIM     = N_MFCC * 3   # mfcc + delta + delta-delta = 24

# Augmentation targets
AUG_TARGET      = {"footsteps": 320, "noise": 280, "balastic": 374}
RMS_SILENCE_THR = 0.001
RMS_VAD_THR     = 0.002

for p in [OUTPUT_PATH, FIGURES_PATH, PROCESSED_AUDIO]:
    p.mkdir(parents=True, exist_ok=True)

print("Pipeline configuration:")
print(f"  SR={SR} Hz | Clip={CLIP_DURATION}s | N_MFCC={N_MFCC} | Feature dim={FEATURE_DIM}")
print(f"  Augmentation targets: {AUG_TARGET}")

## Stage 1 — Quality Filter

Load all raw files and apply three quality gates:
1. **Silent files** — RMS < 0.001 → remove from pipeline
2. **SR mismatch** — flag (will be fixed in Stage 2, not removed)
3. **Clipped files** — flag (will be normalised in Stage 2, not removed)

Output: `quality_df` with per-file flags and a clean `valid_df` for processing.

In [ ]:
records = []
for cls_dir in sorted(DATASET_PATH.iterdir()):
    if not cls_dir.is_dir():
        continue
    label = cls_dir.name
    for wav_path in sorted(cls_dir.rglob("*.wav")):
        try:
            info = sf.info(str(wav_path))
            records.append({
                "path":       str(wav_path),
                "filename":   wav_path.name,
                "label":      label,
                "native_sr":  info.samplerate,
                "duration_s": round(info.duration, 4),
                "channels":   info.channels,
            })
        except Exception as e:
            print(f"  ⚠ Unreadable: {wav_path.name} — {e}")

raw_df = pd.DataFrame(records)
print(f"Total raw files: {len(raw_df)}")
print(raw_df["label"].value_counts().to_string())

In [ ]:
quality_flags = []

print("Running quality gate...")
for _, row in tqdm(raw_df.iterrows(), total=len(raw_df)):
    try:
        y, sr = librosa.load(row["path"], sr=None, mono=True)
        rms_val    = float(np.sqrt(np.mean(y ** 2)))
        clip_ratio = float(np.mean(np.abs(y) > 0.99))
        quality_flags.append({
            "path":        row["path"],
            "filename":    row["filename"],
            "label":       row["label"],
            "native_sr":   sr,
            "rms":         round(rms_val, 6),
            "clip_ratio":  round(clip_ratio, 5),
            "is_silent":   rms_val < RMS_SILENCE_THR,
            "is_clipped":  clip_ratio > 0.01,
            "sr_mismatch": sr != SR,
        })
    except Exception as e:
        quality_flags.append({
            "path": row["path"], "filename": row["filename"],
            "label": row["label"], "native_sr": None,
            "rms": 0, "clip_ratio": 0,
            "is_silent": True, "is_clipped": False, "sr_mismatch": True,
        })

qual_df = pd.DataFrame(quality_flags)

print("\n── Quality Gate Summary ─────────────────────────────────")
print(f"  Total files       : {len(qual_df)}")
print(f"  Silent (REMOVED)  : {qual_df['is_silent'].sum()}")
print(f"  SR mismatch       : {qual_df['sr_mismatch'].sum()}  → will resample")
print(f"  Clipped           : {qual_df['is_clipped'].sum()}   → will normalise")
print("─────────────────────────────────────────────────────────")

# Remove silent files
valid_df = qual_df[~qual_df["is_silent"]].reset_index(drop=True)
removed_df = qual_df[qual_df["is_silent"]]

print(f"\nFiles removed (silent): {len(removed_df)}")
for _, r in removed_df.iterrows():
    print(f"  ✗ [{r['label']}] {r['filename']}  (RMS={r['rms']:.6f})")

print(f"\nFiles proceeding to Stage 2: {len(valid_df)}")
print(valid_df["label"].value_counts().to_string())

## Stage 2 — Canonical Preprocessing

Every valid file is processed through a deterministic pipeline:



The processed audio files serve as the **reproducible clean base** for all feature extraction and augmentation.

In [ ]:
def preprocess_audio(src_path: str, label: str, is_clipped: bool) -> np.ndarray:
    """Load, resample, normalise, trim/pad → return float32 array of CLIP_SAMPLES."""
    y, sr = librosa.load(src_path, sr=SR, mono=True)

    # Peak normalise (always — safe for clipped AND clean)
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y / peak * 0.95   # leave 5% headroom

    # Trim silence at edges (aggressive only for footsteps — sparse transients)
    if label == "footsteps":
        y, _ = librosa.effects.trim(y, top_db=40)

    # Hard trim or zero-pad to CLIP_SAMPLES
    if len(y) >= CLIP_SAMPLES:
        # Centre crop for footsteps (keep burst), front crop for others
        if label == "footsteps":
            start = max(0, (len(y) - CLIP_SAMPLES) // 2)
        else:
            start = 0
        y = y[start: start + CLIP_SAMPLES]
    else:
        pad_len = CLIP_SAMPLES - len(y)
        y = np.pad(y, (0, pad_len), mode="constant")

    return y.astype(np.float32)

In [ ]:
processed_records = []
fail_count = 0

print("Preprocessing all valid files...")
for _, row in tqdm(valid_df.iterrows(), total=len(valid_df)):
    try:
        y_clean = preprocess_audio(row["path"], row["label"], row["is_clipped"])

        # Save processed wav
        label_dir = PROCESSED_AUDIO / row["label"]
        label_dir.mkdir(parents=True, exist_ok=True)
        out_path = label_dir / row["filename"]
        sf.write(str(out_path), y_clean, SR, subtype="PCM_16")

        processed_records.append({
            "path":       str(out_path),
            "filename":   row["filename"],
            "label":      row["label"],
            "rms_before": row["rms"],
            "rms_after":  float(np.sqrt(np.mean(y_clean ** 2))),
            "clipped":    row["is_clipped"],
            "sr_fixed":   row["sr_mismatch"],
            "split":      "raw",   # will be assigned in Stage 5
        })
    except Exception as e:
        fail_count += 1
        print(f"  ✗ Failed: {row['filename']} — {e}")

proc_df = pd.DataFrame(processed_records)
print(f"\n Processed: {len(proc_df)} files  |  Failed: {fail_count}")
print(proc_df["label"].value_counts().to_string())
print(f"\nAll processed audio saved to: {PROCESSED_AUDIO}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# ── A: RMS before vs after normalisation ────────────────────────────────────
ax = axes[0]
for label, color in CLASS_COLORS.items():
    sub = proc_df[proc_df["label"] == label]
    if sub.empty: continue
    ax.scatter(sub["rms_before"], sub["rms_after"], color=color, alpha=0.55,
               s=20, edgecolors="none", label=label.capitalize())
lim_max = max(proc_df["rms_before"].max(), proc_df["rms_after"].max()) * 1.05
ax.plot([0, lim_max], [0, lim_max], "k--", linewidth=1.0, label="y = x")
ax.set_xlabel("RMS Before Normalisation")
ax.set_ylabel("RMS After Normalisation")
ax.set_title("(a) RMS Normalisation Effect")
ax.legend(markerscale=1.8)
paper_axes(ax)

# ── B: Duration before vs after (seconds) ───────────────────────────────────
ax = axes[1]
raw_dur_by_class = {r["label"]: [] for r in processed_records}
for r in processed_records:
    raw_dur_by_class[r["label"]].append(
        valid_df[valid_df["filename"] == r["filename"]]["duration_s"].values[0]
        if r["filename"] in valid_df["filename"].values else CLIP_DURATION
    )

classes = [c for c in CLASS_COLORS if c in raw_dur_by_class]
data_before = [raw_dur_by_class[c] for c in classes]

bp = ax.boxplot(data_before, patch_artist=True, notch=False,
                medianprops=dict(color="black", linewidth=2),
                positions=np.arange(len(classes)) * 2 - 0.35, widths=0.6)
for patch, cls in zip(bp["boxes"], classes):
    patch.set_facecolor(CLASS_COLORS[cls])
    patch.set_alpha(0.5)

ax.axhline(CLIP_DURATION, color="red", linewidth=1.5, linestyle="--",
           label=f"Target = {CLIP_DURATION}s")
ax.set_xticks(np.arange(len(classes)) * 2 - 0.35)
ax.set_xticklabels([c.capitalize() for c in classes])
ax.set_ylabel("Duration (seconds)")
ax.set_title("(b) Pre-processing Duration Distribution")
ax.legend()
paper_axes(ax)

# ── C: SR fix summary ────────────────────────────────────────────────────────
ax = axes[2]
sr_fixed   = proc_df[proc_df["sr_fixed"] == True]["label"].value_counts()
sr_ok      = proc_df[proc_df["sr_fixed"] == False]["label"].value_counts()
clip_fixed = proc_df[proc_df["clipped"] == True]["label"].value_counts()
all_labels = list(CLASS_COLORS.keys())

x = np.arange(len(all_labels))
w = 0.28
ax.bar(x - w,  [sr_fixed.get(l, 0)   for l in all_labels], width=w,
       label="SR Resampled",   color="#4393c3", edgecolor="black", linewidth=0.7)
ax.bar(x,      [clip_fixed.get(l, 0) for l in all_labels], width=w,
       label="Peak Normalised", color="#d6604d", edgecolor="black", linewidth=0.7)
ax.bar(x + w,  [sr_ok.get(l, 0)      for l in all_labels], width=w,
       label="Clean (no fix)",  color="#4dac26", edgecolor="black", linewidth=0.7)

ax.set_xticks(x)
ax.set_xticklabels([l.capitalize() for l in all_labels])
ax.set_ylabel("File Count")
ax.set_title("(c) Preprocessing Corrections Applied")
ax.legend()
paper_axes(ax)

plt.suptitle("Stage 2 — Canonical Preprocessing Validation", fontsize=13,
             fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(str(FIGURES_PATH / "fig_preprocessing_validation.pdf"))
plt.show()
print("Saved → fig_preprocessing_validation.pdf")